In [19]:
#문제1
#보스턴 집값데이터(boston_house)를 가지고 우리가 만든 미분함수를 사용하여 
# 집값을 예측하는 모델을 구현하고 마지막에 정확도를 출력하시오 
#정확도는 r2_score()를 사용한다
#참고)데이터에서 Target이 정답
#학습데이와 테스터데이터 비율은 8:2
#정확도는 테스트 데이터 기준으로 제출한다.

In [20]:
# 데이터 불러오기 

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

# pandas로 불러오기 (첫 줄을 컬럼명으로 자동 인식)
df = pd.read_csv("./boston_house.csv")


In [21]:
print(df.shape)

(506, 14)


In [22]:
print(df.isnull().sum()) #컬럼별 결측치개수
print(df.duplicated().sum()) #중복행갯수

AGE        0
B          0
RM         0
CRIM       0
DIS        0
INDUS      0
LSTAT      0
NOX        0
PTRATIO    0
RAD        0
ZN         0
TAX        0
CHAS       0
Target     0
dtype: int64
0


In [23]:
print(df.describe())

              AGE           B          RM        CRIM         DIS       INDUS  \
count  506.000000  506.000000  506.000000  506.000000  506.000000  506.000000   
mean    68.574901  356.674032    6.284634    3.613524    3.795043   11.136779   
std     28.148861   91.294864    0.702617    8.601545    2.105710    6.860353   
min      2.900000    0.320000    3.561000    0.006320    1.129600    0.460000   
25%     45.025000  375.377500    5.885500    0.082045    2.100175    5.190000   
50%     77.500000  391.440000    6.208500    0.256510    3.207450    9.690000   
75%     94.075000  396.225000    6.623500    3.677083    5.188425   18.100000   
max    100.000000  396.900000    8.780000   88.976200   12.126500   27.740000   

            LSTAT         NOX     PTRATIO         RAD          ZN         TAX  \
count  506.000000  506.000000  506.000000  506.000000  506.000000  506.000000   
mean    12.653063    0.554695   18.455534    9.549407   11.363636  408.237154   
std      7.141062    0.1158

In [24]:
# numpy로 불러오기 (첫 줄이 컬럼명이라 skiprows=1 로 건너뜀)
loaded_data = np.loadtxt("./boston_house.csv", delimiter=',', skiprows=1)

x_data = loaded_data[:, 0:-1]   # 마지막 열(Target) 빼고 전부 → 입력 13개
t_data = loaded_data[:, [-1]]   # 마지막 열(Target) → 정답 (2차원 유지)

print("x_data.shape=", x_data.shape, ", t_data.shape=", t_data.shape)
# numpy로 불러오기 (첫 줄이 컬럼명이라 skiprows=1 로 건너뜀)
loaded_data = np.loadtxt("./boston_house.csv", delimiter=',', skiprows=1)

x_data = loaded_data[:, 0:-1]   # 마지막 열(Target) 빼고 전부 → 입력 13개
t_data = loaded_data[:, [-1]]   # 마지막 열(Target) → 정답 (2차원 유지)

print("x_data.shape=", x_data.shape, ", t_data.shape=", t_data.shape)

x_data.shape= (506, 13) , t_data.shape= (506, 1)
x_data.shape= (506, 13) , t_data.shape= (506, 1)


In [25]:
#가중치, 손실함수, 미분함수, 예측함수

np.random.seed(42)
w = np.random.rand(13, 1)   # 입력변수 13개
b = np.random.rand(1)

def loss_func(x, t):
    y = np.dot(x, w) + b
    return np.sum((t - y)**2) / len(x)   # MSE

def numerical_derivative(f, x):  # 수치미분함수
    delta_x = 1e-4
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=["multi_index"], op_flags=["readwrite"])

    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]

        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x)                      # f(x + delta_x)

        x[idx] = float(tmp_val) - delta_x
        fx2 = f(x)                      # f(x - delta_x)

        grad[idx] = (fx1 - fx2) / (2 * delta_x)

        x[idx] = tmp_val
        it.iternext()

    return grad

def predict(x):
    y = np.dot(x, w) + b
    return y

In [26]:
#학습
np.random.seed(42)
w = np.random.rand(13, 1)   # 입력변수 13개
b = np.random.rand(1)
learning_rate = 1e-2   
f = lambda x: loss_func(x_data, t_data)
print("Initial loss value=", loss_func(x_data, t_data))

for step in range(10001):
    w -= learning_rate * numerical_derivative(f, w)
    b -= learning_rate * numerical_derivative(f, b)

    if (step % 1000 == 0):
        print("step=", step, "loss Value=", loss_func(x_data, t_data))

r2_raw = r2_score(t_data, predict(x_data))

Initial loss value= 614764.3144553488
step= 0 loss Value= 23167769017692.926
step= 1000 loss Value= 1.0583840431817165e+36
step= 2000 loss Value= 1.0583840431817165e+36
step= 3000 loss Value= 1.0583840431817165e+36
step= 4000 loss Value= 1.0583840431817165e+36
step= 5000 loss Value= 1.0583840431817165e+36
step= 6000 loss Value= 1.0583840431817165e+36
step= 7000 loss Value= 1.0583840431817165e+36
step= 8000 loss Value= 1.0583840431817165e+36
step= 9000 loss Value= 1.0583840431817165e+36
step= 10000 loss Value= 1.0583840431817165e+36


In [27]:
#정확도 출력
y_pred = predict(x_data)
print("r2_score=", r2_score(t_data, y_pred))

r2_score= -1.2537190330920944e+34


In [28]:
#정확도를 최대한 향상 
#결측치제거 / 이상치제거 /스케일링 (스탠다드스케일러,민맥스스케일러)



In [29]:
# 8:2 분할 + 표준화(스탠다드 스케일링) + 학습
from sklearn.model_selection import train_test_split

x_train, x_test, t_train, t_test = train_test_split(
    x_data, t_data, test_size=0.2, random_state=42)

# 스케일링 기준은 학습데이터로만 계산
mean, std = x_train.mean(axis=0), x_train.std(axis=0)
x_train_s = (x_train - mean) / std
x_test_s  = (x_test  - mean) / std

np.random.seed(42)
w = np.random.rand(13, 1)
b = np.random.rand(1)

learning_rate = 1e-2
f = lambda x: loss_func(x_train_s, t_train)
print("Initial loss value=", loss_func(x_train_s, t_train))

for step in range(10001):
    w -= learning_rate * numerical_derivative(f, w)
    b -= learning_rate * numerical_derivative(f, b)

    if (step % 1000 == 0):
        print("step=", step, "loss Value=", loss_func(x_train_s, t_train))

# 테스트 데이터 기준 정확도
r2_test = r2_score(t_test, predict(x_test_s))
print("test r2_score=", r2_test)

Initial loss value= 620.5639351923269
step= 0 loss Value= 585.1696366589604
step= 1000 loss Value= 21.699543357230432
step= 2000 loss Value= 21.64576104608664
step= 3000 loss Value= 21.64175419655953
step= 4000 loss Value= 21.641439583872128
step= 5000 loss Value= 21.64141486160308
step= 6000 loss Value= 21.64141291890452
step= 7000 loss Value= 21.64141276624546
step= 8000 loss Value= 21.64141275424937
step= 9000 loss Value= 21.641412753306707
step= 10000 loss Value= 21.641412753232633
test r2_score= 0.6687594435020834


In [30]:
#문제2
#당뇨병 데이터(diabetes)를 가지고 우리가 만든 미분함수를 사용하여 당뇨병을 분류하는 
# 모델을 구현하고 마지막에 정확도를 출력하시오 
#정확도는 accuracy_score()를 사용한다
#참고)데이터에서 Outcome이정답

In [31]:
#데이터 불러오기 + 8:2 분할 + 스케일링

import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

loaded_data = np.loadtxt("./diabetes.csv", delimiter=',', skiprows=1)

x_data = loaded_data[:, 0:-1]   # 입력 8개
t_data = loaded_data[:, [-1]]   # 정답 Outcome (0 또는 1)

# 8:2 분할 (stratify: 0/1 비율을 train/test에 똑같이 유지)
x_train, x_test, t_train, t_test = train_test_split(
    x_data, t_data, test_size=0.2, random_state=42, stratify=t_data)

# 스케일링 기준은 학습데이터로만 계산
mean, std = x_train.mean(axis=0), x_train.std(axis=0)
x_train_s = (x_train - mean) / std
x_test_s  = (x_test  - mean) / std

print("train:", x_train.shape, " test:", x_test.shape)

train: (614, 8)  test: (154, 8)


In [32]:
#시그모이드, 손실함수, 미분함수, 예측함수

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def loss_func(x, t):
    delta = 1e-7   # log(0) 방지
    z = np.dot(x, w) + b
    y = sigmoid(z)
    return -np.sum(t*np.log(y+delta) + (1-t)*np.log((1-y)+delta))   # 크로스엔트로피

def numerical_derivative(f, x):  # 수치미분함수
    delta_x = 1e-2
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=["multi_index"], op_flags=["readwrite"])

    while not it.finished:
        idx = it.multi_index
        tmp_val = x[idx]

        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x)                      # f(x + delta_x)

        x[idx] = float(tmp_val) - delta_x
        fx2 = f(x)                      # f(x - delta_x)

        grad[idx] = (fx1 - fx2) / (2 * delta_x)

        x[idx] = tmp_val
        it.iternext()

    return grad

def predict(x):
    z = np.dot(x, w) + b
    y = sigmoid(z)
    result = (y >= 0.5).astype(int)   # 0.5 이상이면 1(당뇨), 아니면 0
    return y, result

In [33]:
#학습

np.random.seed(42)
w = np.random.rand(8, 1)   # 입력변수 8개
b = np.random.rand(1)

learning_rate = 1e-3
f = lambda x: loss_func(x_train_s, t_train)
print("Initial loss value=", loss_func(x_train_s, t_train))

for step in range(3001):
    w -= learning_rate * numerical_derivative(f, w)
    b -= learning_rate * numerical_derivative(f, b)

    if (step % 500 == 0):
        print("step=", step, "loss Value=", loss_func(x_train_s, t_train))

Initial loss value= 478.90886679297205
step= 0 loss Value= 447.95409078401633
step= 500 loss Value= 286.2508772718251
step= 1000 loss Value= 286.2508772718252
step= 1500 loss Value= 286.2508772718252
step= 2000 loss Value= 286.2508772718252
step= 2500 loss Value= 286.2508772718252
step= 3000 loss Value= 286.2508772718252


In [34]:
#정확도 출력 (테스트 데이터 기준)

y, result = predict(x_test_s)
print("test accuracy_score=", accuracy_score(t_test, result))

test accuracy_score= 0.7142857142857143


In [35]:
#제출 형식)
#1110_홍길동_문제1정확도_문제2정확도.ipynb
#1110_홍길동_8.2_7.ipynb
#문제1정확도 문제2정확도를 기입
